In [1]:
# ============================================================
# PS S6E5 - exp_20260520_046_cat_shared001v2_features_optuna_refit_min
# CatBoost shared001v2-style features + 045 Optuna best params refit
#
# Base model structure:
#   042: exp_20260520_042_cat_shared001v2_features_min
#
# Params source:
#   045: exp_20260520_045_cat_shared001v2_features_optuna_search_min
#   best_params_045.json
#
# Purpose:
#   Apply 045 Optuna best CatBoost params to the exact 042 feature/fold/original policy,
#   then generate formal 5-fold OOF / test pred / submission artifacts.
#
# Do not change:
#   - CV split
#   - feature engineering from 042
#   - original rows concat policy
#   - shared004 full-FE blocks except inherited Race_Year group-stat drop
#   - Normalized_TyreLife prohibition
#   - OOF/pred save format
#   - blend evaluation policy
#
# Important:
#   This is NOT a new feature experiment.
#   039 LOG features are NOT added.
#   Race_Year itself is NOT dropped.
#   IsOriginalData is NOT dropped.
#   TyreAgeRatio_str is NOT dropped.
# ============================================================
# 046 change:
#   Base: 042
#   Replace CatBoost hyperparameters with 045 Optuna best params only.
#   Validate whether 045 CV gain survives formal refit and add046 blend.


In [2]:
import os
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss

from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260520_046_cat_shared001v2_features_optuna_refit_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    TARGET_CANDIDATES = ["PitNextLap", "PitNextLab"]
    TARGET = "PitNextLap"
    ID_COL = "id"
    DANGER_COL = "Normalized_TyreLife"

    # 046差分:
    # Base model structure is 042.
    # Keep 042 feature engineering / folds / original-row policy unchanged.
    # Apply 045 Optuna best CatBoost params only.
    # Do NOT add 039 LOG features.
    # Do NOT add new FE.
    # Do NOT drop Race_Year itself.
    # Do NOT drop IsOriginalData.
    # Do NOT drop TyreAgeRatio_str.
    DROP_GROUPSTAT_KEYWORDS = [
        "_by_Race_Year",
    ]
    
    COMP_PATHS = [
        "/kaggle/input/competitions/playground-series-s6e5",
        "/kaggle/input/playground-series-s6e5",
        ".",
    ]

    ORIGINAL_PATHS = [
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "./f1_strategy_dataset_v4.csv",
    ]

    BEST_PARAMS_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/exp_20260520_045_cat_shared001v2_features_optuna_search_min/best_params_045.json"
    with open(BEST_PARAMS_PATH, mode="rt", encoding="utf-8") as f:
        best_params_data = json.load(f)

    # Accept both formats:
    #   1) {"learning_rate": ..., "depth": ...}
    #   2) {"best_params": {"learning_rate": ..., "depth": ...}, ...}
    BEST_PARAMS = best_params_data.get("best_params", best_params_data)
    BEST_PARAMS_SOURCE_RAW = best_params_data

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    SEED = 42
    N_SPLITS = 5

    USE_ORIGINAL = True

    # CatBoost
    TASK_TYPE = "GPU"
    DEVICES = "0:1"

    ITERATIONS = 9000
    LEARNING_RATE = 0.033
    DEPTH = 8
    L2_LEAF_REG = 7.0
    RANDOM_STRENGTH = 0.8
    EARLY_STOPPING_ROUNDS = 350

    # GPU hidden-default guard
    # SlideShare note:
    #   GPU default border_count / bootstrap_type can differ from CPU.
    #   Fix these explicitly.
    BOOTSTRAP_TYPE = "Bernoulli"
    SUBSAMPLE = 0.8
    BORDER_COUNT = 254

    AUTO_CLASS_WEIGHTS = "Balanced"

    CLIP_LOW = 1e-6
    CLIP_HIGH = 1.0 - 1e-6

    VERBOSE = 200


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def print_section(title: str) -> None:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def first_existing_dir(paths):
    for p in paths:
        path = Path(p)
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError(f"No valid competition path found: {paths}")


def first_existing_file(paths, required=True):
    for p in paths:
        path = Path(p)
        if path.exists():
            return path
    if required:
        raise FileNotFoundError(f"No valid file path found: {paths}")
    return None


def resolve_target_column(df: pd.DataFrame) -> str:
    for c in CFG.TARGET_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"No target column found among {CFG.TARGET_CANDIDATES}")


def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in out.columns:
        dtype = out[col].dtype

        if pd.api.types.is_integer_dtype(dtype):
            c_min = out[col].min()
            c_max = out[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                out[col] = out[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                out[col] = out[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                out[col] = out[col].astype(np.int32)

        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")

    return out


def safe_divide(a, b, eps=1e-6):
    return a / (b + eps)


def clip_pred(x):
    return np.clip(x, CFG.CLIP_LOW, CFG.CLIP_HIGH)


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# 015: Race_Year group-stat drop utility
# ============================================================

def is_race_year_groupstat_feature(col: str) -> bool:
    """
    Drop only features generated by add_light_group_stats
    with Race_Year as group key.

    Expected patterns:
      - LapTime_Delta_mean_by_Race_Year
      - LapTime_Delta_std_by_Race_Year
      - LapTime_Delta_diff_mean_by_Race_Year
      - Position_Change_mean_by_Race_Year
      - Position_Change_std_by_Race_Year
      - Position_Change_diff_mean_by_Race_Year
      - RaceProgress_mean_by_Race_Year
      - RaceProgress_std_by_Race_Year
      - RaceProgress_diff_mean_by_Race_Year
      - TyreLife_mean_by_Race_Year
      - TyreLife_std_by_Race_Year
      - TyreLife_diff_mean_by_Race_Year
    """
    return any(keyword in col for keyword in CFG.DROP_GROUPSTAT_KEYWORDS)


def drop_015_groupstat_features(feature_cols):
    """
    015 is a strict ablation from 012.
    Drop only Race_Year group-stat features.

    Do not drop:
      - Race_Year itself
      - Race_Year_count
      - Race_Year_freq
      - IsOriginalData
      - TyreAgeRatio_str
      - Race_Compound_Stint group stats
    """
    return [c for c in feature_cols if not is_race_year_groupstat_feature(c)]


def assert_015_feature_policy(feature_cols):
    """
    Guardrail for 015.
    This experiment is invalid if Race_Year group-stat features remain.
    It is also invalid if Race_Year itself or other key risk features are accidentally dropped.
    """
    remained = [c for c in feature_cols if is_race_year_groupstat_feature(c)]
    assert len(remained) == 0, (
        "015 policy violation: Race_Year group-stat features still remain: "
        + str(remained[:20])
    )

    assert "Race_Year" in feature_cols, (
        "015 policy violation: Race_Year itself was dropped. "
        "015 should drop only Race_Year group stats."
    )

    assert "IsOriginalData" in feature_cols, (
        "015 policy violation: IsOriginalData was dropped. "
        "015 should keep IsOriginalData."
    )

    assert "TyreAgeRatio_str" in feature_cols, (
        "015 policy violation: TyreAgeRatio_str was dropped. "
        "015 should keep TyreAgeRatio_str."
    )

In [6]:
# ============================================================
# Load data
# ============================================================

print_section("Load Data")

COMP_PATH = first_existing_dir(CFG.COMP_PATHS)

train_raw = pd.read_csv(COMP_PATH / "train.csv")
test_raw = pd.read_csv(COMP_PATH / "test.csv")
sample_submission = pd.read_csv(COMP_PATH / "sample_submission.csv")

target_col_train = resolve_target_column(train_raw)
if target_col_train != CFG.TARGET:
    train_raw = train_raw.rename(columns={target_col_train: CFG.TARGET})

target_col_sub = resolve_target_column(sample_submission)
if target_col_sub != CFG.TARGET:
    sample_submission = sample_submission.rename(columns={target_col_sub: CFG.TARGET})

original_raw = None
original_path = first_existing_file(CFG.ORIGINAL_PATHS, required=False)

if CFG.USE_ORIGINAL and original_path is not None:
    original_raw = pd.read_csv(original_path)

    if CFG.TARGET not in original_raw.columns:
        original_target = resolve_target_column(original_raw)
        original_raw = original_raw.rename(columns={original_target: CFG.TARGET})

    print("Loaded original:", original_path, original_raw.shape)

print("COMP_PATH:", COMP_PATH)
print("train_raw:", train_raw.shape)
print("test_raw :", test_raw.shape)
print("sample_submission:", sample_submission.shape)
print("target mean train:", train_raw[CFG.TARGET].mean())

assert CFG.ID_COL in train_raw.columns
assert CFG.ID_COL in test_raw.columns
assert CFG.ID_COL in sample_submission.columns
assert CFG.TARGET in train_raw.columns
assert CFG.TARGET in sample_submission.columns
assert test_raw[CFG.ID_COL].equals(sample_submission[CFG.ID_COL])

if original_raw is not None:
    assert CFG.TARGET in original_raw.columns
    print("target mean original:", original_raw[CFG.TARGET].mean())

    if CFG.DANGER_COL in original_raw.columns:
        original_raw = original_raw.drop(columns=[CFG.DANGER_COL])
        print(f"Dropped danger column from original: {CFG.DANGER_COL}")

train_raw = reduce_mem_usage(train_raw)
test_raw = reduce_mem_usage(test_raw)

if original_raw is not None:
    original_raw = reduce_mem_usage(original_raw)

train_ids = train_raw[CFG.ID_COL].copy()
test_ids = test_raw[CFG.ID_COL].copy()


Load Data
Loaded original: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv (101371, 16)
COMP_PATH: /kaggle/input/competitions/playground-series-s6e5
train_raw: (439140, 16)
test_raw : (188165, 15)
sample_submission: (188165, 2)
target mean train: 0.19898210137996994
target mean original: 0.25479673673930414
Dropped danger column from original: Normalized_TyreLife


In [7]:
# ============================================================
# shared_004 style feature engineering
# ============================================================

BASE_CAT_COLS = ["Driver", "Compound", "Race"]

NUMERIC_BASE_COLS = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]


def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6

    for col in BASE_CAT_COLS:
        if col in out.columns:
            out[col] = out[col].astype("string").fillna("__MISSING__").astype(str)

    if {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedTotalLaps"] = safe_divide(
            out["LapNumber"],
            out["RaceProgress"].clip(lower=eps),
            eps,
        ).replace([np.inf, -np.inf], np.nan).clip(1, 120)

        out["LapsRemaining"] = (
            out["EstimatedTotalLaps"] - out["LapNumber"]
        ).clip(lower=0)

        out["LapProgress_x_LapNumber"] = out["LapNumber"] * out["RaceProgress"]

    if {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreAgeRatio"] = safe_divide(out["TyreLife"], out["LapNumber"].clip(lower=1), eps)
        out["LapPerTyreLife"] = safe_divide(out["LapNumber"], out["TyreLife"] + 1, eps)
        out["LapMinusTyreLife"] = out["LapNumber"] - out["TyreLife"]
        out["TyreLifeMinusLap"] = out["TyreLife"] - out["LapNumber"]

    if {"TyreLife", "EstimatedTotalLaps"}.issubset(out.columns):
        out["TyreAgeVsRace"] = safe_divide(
            out["TyreLife"],
            out["EstimatedTotalLaps"].clip(lower=1),
            eps,
        )

    if {"TyreLife", "RaceProgress"}.issubset(out.columns):
        out["PitWindowPressure"] = out["TyreLife"] * out["RaceProgress"]
        out["TyreLife_x_RaceProgress"] = out["TyreLife"] * out["RaceProgress"]

    if {"TyreLife", "LapsRemaining"}.issubset(out.columns):
        out["TyreLife_to_LapsRemaining"] = safe_divide(out["TyreLife"], out["LapsRemaining"] + 1, eps)
        out["LapsRemaining_to_TyreLife"] = safe_divide(out["LapsRemaining"], out["TyreLife"] + 1, eps)

    if {"Cumulative_Degradation", "TyreLife"}.issubset(out.columns):
        out["DegPerTyreLap"] = safe_divide(
            out["Cumulative_Degradation"],
            out["TyreLife"].clip(lower=1),
            eps,
        )
        out["AbsDegPerTyreLap"] = safe_divide(
            out["Cumulative_Degradation"].abs(),
            out["TyreLife"].clip(lower=1),
            eps,
        )

    if {"Cumulative_Degradation", "LapNumber"}.issubset(out.columns):
        out["DegPerRaceLap"] = safe_divide(
            out["Cumulative_Degradation"],
            out["LapNumber"].clip(lower=1),
            eps,
        )

    if {"LapTime_Delta", "TyreLife"}.issubset(out.columns):
        out["DeltaPerTyreLap"] = safe_divide(
            out["LapTime_Delta"],
            out["TyreLife"].clip(lower=1),
            eps,
        )
        out["AbsDeltaPerTyreLap"] = safe_divide(
            out["LapTime_Delta"].abs(),
            out["TyreLife"].clip(lower=1),
            eps,
        )

    if "LapTime_Delta" in out.columns:
        out["DeltaAbs"] = out["LapTime_Delta"].abs()
        out["LapTimeDeltaPositive"] = (out["LapTime_Delta"] > 0).astype(np.int8)
        out["LapTimeDeltaNegative"] = (out["LapTime_Delta"] < 0).astype(np.int8)

    if "Cumulative_Degradation" in out.columns:
        out["Abs_Cumulative_Degradation"] = out["Cumulative_Degradation"].abs()
        out["Positive_Degradation"] = (out["Cumulative_Degradation"] > 0).astype(np.int8)

    if "Position_Change" in out.columns:
        out["Abs_Position_Change"] = out["Position_Change"].abs()
        out["Gained_Position"] = (out["Position_Change"] > 0).astype(np.int8)
        out["Lost_Position"] = (out["Position_Change"] < 0).astype(np.int8)

    if {"Position", "RaceProgress"}.issubset(out.columns):
        out["PositionPressure"] = out["Position"] * out["RaceProgress"]

    if {"Stint", "TyreLife"}.issubset(out.columns):
        out["StintPressure"] = out["Stint"] * out["TyreLife"]
        out["TyreLife_x_Stint"] = out["TyreLife"] * out["Stint"]

    if {"Stint", "LapNumber"}.issubset(out.columns):
        out["Stint_x_LapNumber"] = out["Stint"] * out["LapNumber"]
        out["Is_First_Stint"] = (out["Stint"] == 1).astype(np.int8)
        out["Is_Late_Stint"] = (out["Stint"] >= 3).astype(np.int8)

    if "RaceProgress" in out.columns:
        out["Early_Race"] = (out["RaceProgress"] <= 0.25).astype(np.int8)
        out["Mid_Race"] = (
            (out["RaceProgress"] > 0.25)
            & (out["RaceProgress"] <= 0.65)
        ).astype(np.int8)
        out["Late_Race"] = (out["RaceProgress"] > 0.65).astype(np.int8)

        out["RacePhase"] = pd.cut(
            out["RaceProgress"],
            bins=[-np.inf, 0.20, 0.40, 0.60, 0.80, np.inf],
            labels=["P1", "P2", "P3", "P4", "P5"],
        ).astype(str)

    if "LapNumber" in out.columns:
        out["LapBin"] = pd.cut(
            out["LapNumber"],
            bins=[-np.inf, 5, 10, 20, 35, 50, np.inf],
            labels=[
                "L_000_005",
                "L_006_010",
                "L_011_020",
                "L_021_035",
                "L_036_050",
                "L_051_plus",
            ],
        ).astype(str)

    if "TyreLife" in out.columns:
        out["TyreLifeBin"] = pd.cut(
            out["TyreLife"],
            bins=[-np.inf, 3, 7, 12, 20, 30, np.inf],
            labels=[
                "T_000_003",
                "T_004_007",
                "T_008_012",
                "T_013_020",
                "T_021_030",
                "T_031_plus",
            ],
        ).astype(str)

    if "Position" in out.columns:
        out["PositionBin"] = pd.cut(
            out["Position"],
            bins=[-np.inf, 3, 8, 14, np.inf],
            labels=["front", "upper_mid", "lower_mid", "back"],
        ).astype(str)

    def make_cross(name, cols):
        if set(cols).issubset(out.columns):
            val = out[cols[0]].astype(str)
            for c in cols[1:]:
                val = val + "_" + out[c].astype(str)
            out[name] = val

    make_cross("Race_Year", ["Race", "Year"])
    make_cross("Compound_Stint", ["Compound", "Stint"])
    make_cross("Driver_Race", ["Driver", "Race"])
    make_cross("Driver_Compound", ["Driver", "Compound"])
    make_cross("Race_Compound", ["Race", "Compound"])
    make_cross("Race_Compound_Stint", ["Race", "Compound", "Stint"])
    make_cross("Compound_RacePhase", ["Compound", "RacePhase"])
    make_cross("Compound_TyreLifeBin", ["Compound", "TyreLifeBin"])
    make_cross("RacePhase_TyreLifeBin", ["RacePhase", "TyreLifeBin"])

    out = out.replace([np.inf, -np.inf], np.nan)

    for col in out.select_dtypes(include=["float64"]).columns:
        out[col] = out[col].astype(np.float32)

    return out


def add_digit_features(df: pd.DataFrame, numeric_cols=None, int_digit_limit=3, decimal_digit_limit=2) -> pd.DataFrame:
    out = df.copy()

    if numeric_cols is None:
        numeric_cols = out.select_dtypes(include=[np.number]).columns.tolist()

    for col in numeric_cols:
        if col not in out.columns:
            continue

        s = out[col].fillna(0).astype(float)
        abs_s = s.abs()

        for i in range(int_digit_limit):
            new_col = f"{col}_int_digit_{i + 1}"
            out[new_col] = ((abs_s // (10 ** i)) % 10).astype(np.int8)

        if pd.api.types.is_float_dtype(out[col]):
            for i in range(1, decimal_digit_limit + 1):
                new_col = f"{col}_dec_digit_{i}"
                out[new_col] = ((abs_s * (10 ** i)).round().astype(int) % 10).astype(np.int8)

    return out


def add_float_signature_features(df: pd.DataFrame, float_cols=None) -> pd.DataFrame:
    out = df.copy()

    selected = [
        "RaceProgress",
        "LapTime (s)",
        "LapTime_Delta",
        "Cumulative_Degradation",
        "TyreAgeRatio",
        "DegPerTyreLap",
        "DegPerRaceLap",
        "DeltaPerTyreLap",
        "DeltaAbs",
        "PitWindowPressure",
        "EstimatedTotalLaps",
        "LapsRemaining",
        "LapMinusTyreLife",
    ]

    float_cols = [c for c in selected if c in out.columns]

    for col in float_cols:
        scaled = (out[col].fillna(0).astype(float) * 100).round().astype(int).abs()

        for i in range(5):
            new_col = f"{col}_sig_{i + 1}"
            digit = ((scaled // (10 ** i)) % 10).astype(np.int8)

            if digit.nunique() > 1:
                out[new_col] = digit.astype(str)

    return out


def add_string_precision_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "RaceProgress" in out.columns:
        out["RaceProgress_str"] = out["RaceProgress"].round(4).astype(str)

    if "EstimatedTotalLaps" in out.columns:
        out["EstimatedTotalLaps_str"] = out["EstimatedTotalLaps"].round(1).astype(str)

    if "TyreAgeRatio" in out.columns:
        out["TyreAgeRatio_str"] = out["TyreAgeRatio"].round(3).astype(str)

    return out


# ============================================================
# 042: shared001v2-style features from 029/038 RealMLP branch
# ============================================================

SHARED001V2_042_NUMERIC_SOURCE_COLS = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]

SHARED001V2_042_CAT_FEATURES = [
    "PitStop_cat_",
    "Race_Year_",
    "Race_Compound_",
    "RaceProgress_200_quantile_bin_",
    "LapTime (s)_7_quantile_bin_",
]

SHARED001V2_042_COUNT_FEATURES = [
    "_PitStop_cat_count",
    "_Race_Year_count",
    "_Race_Compound_count",
    "_RaceProgress_200_quantile_bin_count",
    "_LapTime_s_7_quantile_bin_count",
]


def _to_safe_str_series(s: pd.Series) -> pd.Series:
    return s.astype("string").fillna("__MISSING__").astype(str)


def _floor_cat_series(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    # Use a dedicated missing token before flooring to avoid silent imputation.
    floored = np.floor(x).astype("float")
    return floored.astype("Int64").astype("string").fillna("__MISSING__").astype(str)


def add_shared001v2_style_features_042(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add 029/038 RealMLP-style categorical/bin features to CatBoost.
    These are row-wise transformations only; no target is used here.

    Important:
    - 039 LOG features are intentionally not added.
    - Normalized_TyreLife and any endpoint proxy are not reconstructed.
    """
    out = df.copy()

    if "PitStop" in out.columns:
        out["PitStop_cat_"] = _floor_cat_series(out["PitStop"])

    if {"Race", "Year"}.issubset(out.columns):
        out["Race_Year_"] = _to_safe_str_series(out["Race"]) + "_" + _to_safe_str_series(out["Year"])

    if {"Race", "Compound"}.issubset(out.columns):
        out["Race_Compound_"] = _to_safe_str_series(out["Race"]) + "_" + _to_safe_str_series(out["Compound"])

    # 029/038-style quantile bins.
    # qcut is avoided because train/test/original must receive the same deterministic mapping.
    if "RaceProgress" in out.columns:
        rp = pd.to_numeric(out["RaceProgress"], errors="coerce").clip(0, 1)
        bin_idx = np.floor(rp.fillna(-1) * 200).astype(int).clip(-1, 199)
        out["RaceProgress_200_quantile_bin_"] = np.where(
            bin_idx < 0,
            "__MISSING__",
            "rp200_" + pd.Series(bin_idx, index=out.index).astype(str),
        )

    if "LapTime (s)" in out.columns:
        lt = pd.to_numeric(out["LapTime (s)"], errors="coerce")
        # Use deterministic absolute bins rather than qcut.
        # qcut per-frame would make train/test/original bin definitions inconsistent.
        codes = pd.cut(
            lt,
            bins=[-np.inf, 70, 75, 80, 85, 90, 95, np.inf],
            labels=False,
        )
        out["LapTime (s)_7_quantile_bin_"] = (
            "lt7_" + codes.astype("Int64").astype("string")
        ).fillna("__MISSING__").astype(str)

    for col in SHARED001V2_042_NUMERIC_SOURCE_COLS:
        if col in out.columns:
            out[f"{col}_floor_cat_042"] = _floor_cat_series(out[col])

    return out


def add_shared001v2_counts_042(train_df, test_df, original_df=None):
    """
    Count-encode only the newly added shared001v2-style categorical/bin columns.
    Counts are calculated from train + test + original feature space, without target use.
    """
    count_source_cols = [
        "PitStop_cat_",
        "Race_Year_",
        "Race_Compound_",
        "RaceProgress_200_quantile_bin_",
        "LapTime (s)_7_quantile_bin_",
    ] + [
        f"{c}_floor_cat_042"
        for c in SHARED001V2_042_NUMERIC_SOURCE_COLS
        if f"{c}_floor_cat_042" in train_df.columns and f"{c}_floor_cat_042" in test_df.columns
    ]

    count_source_cols = [
        c for c in count_source_cols
        if c in train_df.columns and c in test_df.columns
    ]

    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    base = pd.concat(
        [f[count_source_cols].astype(str) for f in frames],
        axis=0,
        ignore_index=True,
    )
    total = len(base)

    for col in count_source_cols:
        counts = base[col].astype(str).value_counts(dropna=False)
        safe_name = col.replace(" ", "_").replace("(", "").replace(")", "")
        for f in frames:
            f[f"_{safe_name}_count"] = f[col].astype(str).map(counts).fillna(0).astype(np.float32)
            f[f"_{safe_name}_freq"] = (f[f"_{safe_name}_count"] / total).astype(np.float32)

    del base
    gc.collect()

    if original_df is not None:
        return train_df, test_df, original_df

    return train_df, test_df, None



def add_frequency_features(train_df, test_df, original_df=None):
    freq_cols = [
        "Driver",
        "Race",
        "Compound",
        "Race_Year",
        "Compound_Stint",
        "Driver_Race",
        "Driver_Compound",
        "Race_Compound",
        "Race_Compound_Stint",
        "Compound_RacePhase",
        "Compound_TyreLifeBin",
        "RacePhase_TyreLifeBin",
        "LapBin",
        "TyreLifeBin",
        "PositionBin",
    ]

    freq_cols = [c for c in freq_cols if c in train_df.columns and c in test_df.columns]

    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    base = pd.concat([f[freq_cols] for f in frames], axis=0, ignore_index=True)
    total = len(base)

    for col in freq_cols:
        counts = base[col].astype(str).value_counts(dropna=False)

        for f in frames:
            f[f"{col}_count"] = f[col].astype(str).map(counts).fillna(0).astype(np.float32)
            f[f"{col}_freq"] = (f[f"{col}_count"] / total).astype(np.float32)

    del base
    gc.collect()

    if original_df is not None:
        return train_df, test_df, original_df

    return train_df, test_df, None


def add_light_group_stats(train_df, test_df, original_df=None):
    group_cols_list = [
        ["Race_Year"],
        ["Race_Compound_Stint"],
        ["Driver_Race"],
        ["Compound_Stint"],
    ]

    value_cols = [
        "LapTime_Delta",
        "Position_Change",
        "RaceProgress",
        "TyreLife",
    ]

    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    for group_cols in group_cols_list:
        group_cols = [c for c in group_cols if all(c in f.columns for f in frames)]

        if not group_cols:
            continue

        group_name = "_".join(group_cols)

        for value_col in value_cols:
            if not all(value_col in f.columns for f in frames):
                continue

            base = pd.concat(
                [f[group_cols + [value_col]] for f in frames],
                axis=0,
                ignore_index=True,
            )

            stats = (
                base.groupby(group_cols, dropna=False)[value_col]
                .agg(["mean", "std"])
                .reset_index()
            )

            stats.columns = group_cols + [
                f"{value_col}_mean_by_{group_name}",
                f"{value_col}_std_by_{group_name}",
            ]

            for idx, f in enumerate(frames):
                frames[idx] = f.merge(stats, on=group_cols, how="left")

                mean_col = f"{value_col}_mean_by_{group_name}"
                frames[idx][f"{value_col}_diff_mean_by_{group_name}"] = (
                    frames[idx][value_col] - frames[idx][mean_col]
                ).astype(np.float32)

            del base, stats
            gc.collect()

    if original_df is not None:
        return frames[0], frames[1], frames[2]

    return frames[0], frames[1], None


def clean_columns(train_df, test_df, original_df=None):
    train_df = train_df.copy()
    test_df = test_df.copy()

    if original_df is not None:
        original_df = original_df.copy()

    common_cols = [c for c in train_df.columns if c in test_df.columns]

    train_df = train_df[common_cols + [CFG.TARGET]]
    test_df = test_df[common_cols]

    if original_df is not None:
        for c in common_cols:
            if c not in original_df.columns:
                original_df[c] = np.nan

        original_df = original_df[common_cols + [CFG.TARGET]]

    return train_df, test_df, original_df


def fill_missing_and_types(train_df, test_df, original_df=None):
    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    all_feature_cols = [c for c in train_df.columns if c != CFG.TARGET]

    cat_cols = []

    for col in all_feature_cols:
        if any(
            (
                f[col].dtype == "object"
                or str(f[col].dtype).startswith("category")
                or str(f[col].dtype).startswith("string")
            )
            for f in frames
            if col in f.columns
        ):
            cat_cols.append(col)

    num_cols = [c for c in all_feature_cols if c not in cat_cols]

    for col in cat_cols:
        values = pd.concat(
            [f[col].astype("string") for f in frames if col in f.columns],
            axis=0,
        )
        mode_values = values.mode()
        mode_value = mode_values.iloc[0] if len(mode_values) else "__MISSING__"

        for f in frames:
            if col in f.columns:
                f[col] = f[col].astype("string").fillna(mode_value).astype(str)

    for col in num_cols:
        values = pd.concat([f[col] for f in frames if col in f.columns], axis=0)
        fill_value = values.median()

        for f in frames:
            if col in f.columns:
                f[col] = f[col].replace([np.inf, -np.inf], np.nan).fillna(fill_value)
                if f[col].dtype == "float64":
                    f[col] = f[col].astype(np.float32)

    train_df = reduce_mem_usage(train_df)
    test_df = reduce_mem_usage(test_df)

    if original_df is not None:
        original_df = reduce_mem_usage(original_df)

    return train_df, test_df, original_df, cat_cols

In [8]:
# ============================================================
# Feature Engineering
# ============================================================

print_section("Feature Engineering")

train_fe = train_raw.copy()
test_fe = test_raw.copy()

if original_raw is not None and CFG.TARGET in original_raw.columns:
    original_fe = original_raw.copy()
else:
    original_fe = None

if original_fe is not None and CFG.DANGER_COL in original_fe.columns:
    original_fe = original_fe.drop(columns=[CFG.DANGER_COL])
    print(f"Dropped danger column from original_fe: {CFG.DANGER_COL}")

train_fe["IsOriginalData"] = 0
test_fe["IsOriginalData"] = 0

if original_fe is not None:
    original_fe["IsOriginalData"] = 1

train_fe = add_domain_features(train_fe)
test_fe = add_domain_features(test_fe)

if original_fe is not None:
    original_fe = add_domain_features(original_fe)

digit_source_cols = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
    "EstimatedTotalLaps",
    "LapsRemaining",
    "TyreAgeRatio",
    "DegPerTyreLap",
    "DegPerRaceLap",
    "DeltaPerTyreLap",
    "DeltaAbs",
    "PositionPressure",
    "StintPressure",
    "PitWindowPressure",
    "LapMinusTyreLife",
]
digit_source_cols = [c for c in digit_source_cols if c in train_fe.columns and c in test_fe.columns]

train_fe = add_digit_features(train_fe, digit_source_cols)
test_fe = add_digit_features(test_fe, digit_source_cols)

if original_fe is not None:
    original_fe = add_digit_features(original_fe, digit_source_cols)

train_fe = add_float_signature_features(train_fe)
test_fe = add_float_signature_features(test_fe)

if original_fe is not None:
    original_fe = add_float_signature_features(original_fe)

train_fe = add_string_precision_features(train_fe)
test_fe = add_string_precision_features(test_fe)

if original_fe is not None:
    original_fe = add_string_precision_features(original_fe)

# 042: add shared001v2-style categorical/bin/count features.
# Keep this before fill_missing/types so CatBoost receives these as categorical columns.
train_fe = add_shared001v2_style_features_042(train_fe)
test_fe = add_shared001v2_style_features_042(test_fe)

if original_fe is not None:
    original_fe = add_shared001v2_style_features_042(original_fe)

train_fe, test_fe, original_fe = add_frequency_features(train_fe, test_fe, original_fe)
train_fe, test_fe, original_fe = add_shared001v2_counts_042(train_fe, test_fe, original_fe)
train_fe, test_fe, original_fe = add_light_group_stats(train_fe, test_fe, original_fe)

train_fe, test_fe, original_fe = clean_columns(train_fe, test_fe, original_fe)
train_fe, test_fe, original_fe, cat_cols = fill_missing_and_types(train_fe, test_fe, original_fe)

print("train_fe:", train_fe.shape)
print("test_fe :", test_fe.shape)

if original_fe is not None:
    print("original_fe:", original_fe.shape)

print("categorical features:", len(cat_cols))


Feature Engineering
train_fe: (439140, 351)
test_fe : (188165, 350)
original_fe: (101371, 351)
categorical features: 93


In [9]:
# ============================================================
# Prepare matrices
# ============================================================

print_section("Prepare Matrices")

X_comp = train_fe.drop(columns=[CFG.TARGET, CFG.ID_COL], errors="ignore")
y_comp = train_fe[CFG.TARGET].astype(int).reset_index(drop=True)

X_test_final = test_fe.drop(columns=[CFG.ID_COL], errors="ignore")

if original_fe is not None:
    X_orig = original_fe.drop(columns=[CFG.TARGET, CFG.ID_COL], errors="ignore")
    y_orig = original_fe[CFG.TARGET].astype(int).reset_index(drop=True)
else:
    X_orig = None
    y_orig = None

common_features = [c for c in X_comp.columns if c in X_test_final.columns]

# 015差分: Race_Year group-stat featuresだけを除外
dropped_groupstat_features_015 = [
    c for c in common_features
    if is_race_year_groupstat_feature(c)
]

common_features = drop_015_groupstat_features(common_features)
assert_015_feature_policy(common_features)

print("042 inherited-042 inherited-015 dropped Race_Year group-stat features:", dropped_groupstat_features_015)
print("042 inherited-042 inherited-015 dropped feature count:", len(dropped_groupstat_features_015))
print("final feature count after 042 inherited-015 drop:", len(common_features))
print("Race_Year in features:", "Race_Year" in common_features)
print("IsOriginalData in features:", "IsOriginalData" in common_features)
print("TyreAgeRatio_str in features:", "TyreAgeRatio_str" in common_features)
print(
    "Remaining Race_Year group-stat features:",
    [c for c in common_features if is_race_year_groupstat_feature(c)]
)

# 042 guardrails: these shared001v2-style columns should exist unless source columns are missing.
expected_shared001v2_features_042 = [
    c for c in [
        "PitStop_cat_",
        "Race_Year_",
        "Race_Compound_",
        "RaceProgress_200_quantile_bin_",
        "LapTime (s)_7_quantile_bin_",
    ]
    if c in X_comp.columns and c in X_test_final.columns
]
print("042 shared001v2-style base features:", expected_shared001v2_features_042)
print("042 shared001v2-style feature count:", len([
    c for c in common_features
    if c.endswith("_floor_cat_042")
    or c in expected_shared001v2_features_042
    or c.startswith("_PitStop_cat_")
    or c.startswith("_Race_Year_")
    or c.startswith("_Race_Compound_")
    or c.startswith("_RaceProgress_200_quantile_bin_")
    or c.startswith("_LapTime_s_7_quantile_bin_")
]))

X_comp = X_comp[common_features].reset_index(drop=True)
X_test_final = X_test_final[common_features].reset_index(drop=True)

if X_orig is not None:
    for c in common_features:
        if c not in X_orig.columns:
            X_orig[c] = np.nan
    X_orig = X_orig[common_features].reset_index(drop=True)

cat_cols = [c for c in cat_cols if c in common_features]
cat_features = [X_comp.columns.get_loc(c) for c in cat_cols]


Prepare Matrices
042 inherited-042 inherited-015 dropped Race_Year group-stat features: ['LapTime_Delta_mean_by_Race_Year', 'LapTime_Delta_std_by_Race_Year', 'LapTime_Delta_diff_mean_by_Race_Year', 'Position_Change_mean_by_Race_Year', 'Position_Change_std_by_Race_Year', 'Position_Change_diff_mean_by_Race_Year', 'RaceProgress_mean_by_Race_Year', 'RaceProgress_std_by_Race_Year', 'RaceProgress_diff_mean_by_Race_Year', 'TyreLife_mean_by_Race_Year', 'TyreLife_std_by_Race_Year', 'TyreLife_diff_mean_by_Race_Year']
042 inherited-042 inherited-015 dropped feature count: 12
final feature count after 042 inherited-015 drop: 337
Race_Year in features: True
IsOriginalData in features: True
TyreAgeRatio_str in features: True
Remaining Race_Year group-stat features: []
042 shared001v2-style base features: ['PitStop_cat_', 'Race_Year_', 'Race_Compound_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_']
042 shared001v2-style feature count: 26


In [10]:
# ============================================================
# CatBoost model
# ============================================================

def get_catboost_params(seed: int, iterations: int, task_type: str):
    params = {
        "iterations": iterations,
        "learning_rate": CFG.LEARNING_RATE,
        "depth": CFG.DEPTH,
        "l2_leaf_reg": CFG.L2_LEAF_REG,
        "random_strength": CFG.RANDOM_STRENGTH,

        # Fixed to avoid CatBoost CPU/GPU hidden-default mismatch.
        "bootstrap_type": CFG.BOOTSTRAP_TYPE,
        "subsample": CFG.SUBSAMPLE,
        "border_count": CFG.BORDER_COUNT,

        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "auto_class_weights": CFG.AUTO_CLASS_WEIGHTS,

        "task_type": task_type,
        "random_seed": seed,
        "early_stopping_rounds": CFG.EARLY_STOPPING_ROUNDS,
        "allow_writing_files": False,
        "verbose": CFG.VERBOSE,
    }

    params.update(CFG.BEST_PARAMS)

    if task_type == "GPU":
        params["devices"] = CFG.DEVICES
    else:
        params["thread_count"] = -1

    return params


def fit_catboost_with_fallback(X_tr, y_tr, X_va, y_va, seed):
    params = get_catboost_params(seed=seed, iterations=CFG.ITERATIONS, task_type=CFG.TASK_TYPE)

    model = CatBoostClassifier(**params)

    try:
        model.fit(
            X_tr,
            y_tr,
            eval_set=(X_va, y_va),
            cat_features=cat_features,
            use_best_model=True,
        )
        used_device = CFG.TASK_TYPE
        return model, used_device

    except Exception as e:
        print("CatBoost GPU/device training failed. Retrying with CPU.")
        print("Error:", repr(e))

        cpu_params = get_catboost_params(seed=seed, iterations=CFG.ITERATIONS, task_type="CPU")
        cpu_model = CatBoostClassifier(**cpu_params)

        cpu_model.fit(
            X_tr,
            y_tr,
            eval_set=(X_va, y_va),
            cat_features=cat_features,
            use_best_model=True,
        )

        return cpu_model, "CPU"

In [11]:
# ============================================================
# 5fold OOF training
# ============================================================

print_section("5fold OOF Training")

folds = list(
    StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    ).split(X_comp, y_comp)
)

oof = np.zeros(len(X_comp), dtype=np.float32)
pred = np.zeros(len(X_test_final), dtype=np.float32)

fold_records = []
feature_importance_records = []
used_devices = []

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    print("\n" + "=" * 90)
    print(f"Fold {fold}/{CFG.N_SPLITS}")
    print("=" * 90)

    X_tr_comp = X_comp.iloc[tr_idx].copy().reset_index(drop=True)
    y_tr_comp = y_comp.iloc[tr_idx].copy().reset_index(drop=True)

    X_va = X_comp.iloc[va_idx].copy().reset_index(drop=True)
    y_va = y_comp.iloc[va_idx].copy().reset_index(drop=True)

    if X_orig is not None and CFG.USE_ORIGINAL:
        X_tr = pd.concat(
            [X_tr_comp, X_orig],
            axis=0,
            ignore_index=True,
        )
        y_tr = pd.concat(
            [y_tr_comp, y_orig],
            axis=0,
            ignore_index=True,
        )
    else:
        X_tr = X_tr_comp
        y_tr = y_tr_comp

    print("X_tr:", X_tr.shape, "target mean:", float(y_tr.mean()))
    print("X_va:", X_va.shape, "target mean:", float(y_va.mean()))
    print("X_test:", X_test_final.shape)

    model, used_device = fit_catboost_with_fallback(
        X_tr=X_tr,
        y_tr=y_tr,
        X_va=X_va,
        y_va=y_va,
        seed=CFG.SEED + fold,
    )

    va_pred = clip_pred(model.predict_proba(X_va)[:, 1]).astype(np.float32)
    te_pred = clip_pred(model.predict_proba(X_test_final)[:, 1]).astype(np.float32)

    oof[va_idx] = va_pred
    pred += te_pred / CFG.N_SPLITS

    fold_auc = roc_auc_score(y_va, va_pred)
    fold_logloss = log_loss(y_va, va_pred)
    best_iter = int(model.get_best_iteration() or -1)

    print(f"Fold {fold} AUC    : {fold_auc:.9f}")
    print(f"Fold {fold} LogLoss: {fold_logloss:.9f}")
    print(f"Fold {fold} best_iter: {best_iter}")
    print(f"Fold {fold} used_device: {used_device}")

    fold_records.append({
        "fold": fold,
        "auc": float(fold_auc),
        "logloss": float(fold_logloss),
        "best_iteration": best_iter,
        "used_device": used_device,
        "n_train_comp": int(len(X_tr_comp)),
        "n_train_orig": int(len(X_orig)) if X_orig is not None and CFG.USE_ORIGINAL else 0,
        "n_valid": int(len(X_va)),
        "n_features": int(X_tr.shape[1]),
        "n_cat_features": int(len(cat_features)),
    })

    fi = pd.DataFrame({
        "feature": X_tr.columns,
        "importance": model.get_feature_importance(),
        "fold": fold,
    })
    feature_importance_records.append(fi)

    used_devices.append(used_device)

    del model, X_tr_comp, y_tr_comp, X_va, y_va, X_tr, y_tr, va_pred, te_pred
    gc.collect()


5fold OOF Training

Fold 1/5
X_tr: (452683, 337) target mean: 0.21147911452385001
X_va: (87828, 337) target mean: 0.19899121009245344
X_test: (188165, 337)


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9286063	best: 0.9286063 (0)	total: 452ms	remaining: 1h 7m 46s
200:	test: 0.9453290	best: 0.9453290 (200)	total: 46.7s	remaining: 34m 6s
400:	test: 0.9483466	best: 0.9483466 (400)	total: 1m 34s	remaining: 33m 49s
600:	test: 0.9501563	best: 0.9501563 (600)	total: 2m 23s	remaining: 33m 18s
800:	test: 0.9511303	best: 0.9511303 (800)	total: 3m 11s	remaining: 32m 41s
1000:	test: 0.9518530	best: 0.9518530 (1000)	total: 4m 4s	remaining: 32m 34s
1200:	test: 0.9523494	best: 0.9523494 (1200)	total: 4m 58s	remaining: 32m 18s
1400:	test: 0.9526790	best: 0.9526797 (1399)	total: 5m 51s	remaining: 31m 48s
1600:	test: 0.9529119	best: 0.9529119 (1600)	total: 6m 44s	remaining: 31m 9s
1800:	test: 0.9530852	best: 0.9530852 (1800)	total: 7m 38s	remaining: 30m 31s
2000:	test: 0.9532368	best: 0.9532368 (2000)	total: 8m 30s	remaining: 29m 44s
2200:	test: 0.9533818	best: 0.9533818 (2200)	total: 9m 23s	remaining: 29m 1s
2400:	test: 0.9534931	best: 0.9534931 (2400)	total: 10m 15s	remaining: 28m 12s
260

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9211724	best: 0.9211724 (0)	total: 307ms	remaining: 46m 3s
200:	test: 0.9430679	best: 0.9430679 (200)	total: 48.3s	remaining: 35m 12s
400:	test: 0.9464172	best: 0.9464172 (400)	total: 1m 36s	remaining: 34m 26s
600:	test: 0.9482272	best: 0.9482272 (600)	total: 2m 23s	remaining: 33m 25s
800:	test: 0.9492300	best: 0.9492300 (800)	total: 3m 11s	remaining: 32m 38s
1000:	test: 0.9499645	best: 0.9499645 (1000)	total: 4m 3s	remaining: 32m 22s
1200:	test: 0.9504101	best: 0.9504101 (1200)	total: 4m 56s	remaining: 32m 4s
1400:	test: 0.9507436	best: 0.9507436 (1400)	total: 5m 48s	remaining: 31m 30s
1600:	test: 0.9510267	best: 0.9510267 (1600)	total: 6m 41s	remaining: 30m 56s
1800:	test: 0.9512215	best: 0.9512215 (1800)	total: 7m 34s	remaining: 30m 17s
2000:	test: 0.9513866	best: 0.9513867 (1999)	total: 8m 26s	remaining: 29m 32s
2200:	test: 0.9515315	best: 0.9515315 (2200)	total: 9m 19s	remaining: 28m 47s
2400:	test: 0.9516298	best: 0.9516298 (2400)	total: 10m 12s	remaining: 28m 2s
2600:

...

0:	test: 0.9219781	best: 0.9219781 (0)	total: 340ms	remaining: 50m 55s
200:	test: 0.9448592	best: 0.9448592 (200)	total: 48.6s	remaining: 35m 25s
400:	test: 0.9480288	best: 0.9480288 (400)	total: 1m 35s	remaining: 34m 14s
600:	test: 0.9497244	best: 0.9497244 (600)	total: 2m 23s	remaining: 33m 20s
800:	test: 0.9506305	best: 0.9506305 (800)	total: 3m 10s	remaining: 32m 27s
1000:	test: 0.9512540	best: 0.9512540 (1000)	total: 4m 3s	remaining: 32m 26s
1200:	test: 0.9516889	best: 0.9516889 (1200)	total: 4m 57s	remaining: 32m 11s
1400:	test: 0.9519919	best: 0.9519919 (1400)	total: 5m 50s	remaining: 31m 39s
1600:	test: 0.9522020	best: 0.9522020 (1600)	total: 6m 43s	remaining: 31m 2s
1800:	test: 0.9523697	best: 0.9523697 (1800)	total: 7m 35s	remaining: 30m 20s
2000:	test: 0.9525239	best: 0.9525239 (2000)	total: 8m 28s	remaining: 29m 37s
2200:	test: 0.9526748	best: 0.9526756 (2199)	total: 9m 20s	remaining: 28m 51s
2400:	test: 0.9527827	best: 0.9527832 (2398)	total: 10m 12s	remaining: 28m 3s
2600

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9292786	best: 0.9292786 (0)	total: 340ms	remaining: 51m 1s
200:	test: 0.9437318	best: 0.9437318 (200)	total: 48.3s	remaining: 35m 13s
400:	test: 0.9465084	best: 0.9465084 (400)	total: 1m 35s	remaining: 34m 17s
600:	test: 0.9482685	best: 0.9482685 (600)	total: 2m 23s	remaining: 33m 22s
800:	test: 0.9491844	best: 0.9491844 (800)	total: 3m 10s	remaining: 32m 32s
1000:	test: 0.9498970	best: 0.9498970 (1000)	total: 4m 3s	remaining: 32m 24s
1200:	test: 0.9503939	best: 0.9503939 (1200)	total: 4m 55s	remaining: 31m 56s
1400:	test: 0.9507328	best: 0.9507328 (1399)	total: 5m 48s	remaining: 31m 32s
1600:	test: 0.9509824	best: 0.9509824 (1600)	total: 6m 41s	remaining: 30m 53s
1800:	test: 0.9511698	best: 0.9511698 (1800)	total: 7m 33s	remaining: 30m 14s
2000:	test: 0.9513271	best: 0.9513271 (2000)	total: 8m 26s	remaining: 29m 31s
2200:	test: 0.9514717	best: 0.9514717 (2200)	total: 9m 19s	remaining: 28m 49s
2400:	test: 0.9515849	best: 0.9515849 (2400)	total: 10m 12s	remaining: 28m 4s
2600

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9289547	best: 0.9289547 (0)	total: 269ms	remaining: 40m 25s
200:	test: 0.9447229	best: 0.9447229 (200)	total: 48.8s	remaining: 35m 37s
400:	test: 0.9478984	best: 0.9478984 (400)	total: 1m 36s	remaining: 34m 24s
600:	test: 0.9496977	best: 0.9496977 (600)	total: 2m 22s	remaining: 33m 11s
800:	test: 0.9506592	best: 0.9506592 (800)	total: 3m 10s	remaining: 32m 30s
1000:	test: 0.9513823	best: 0.9513823 (1000)	total: 4m 2s	remaining: 32m 17s
1200:	test: 0.9518432	best: 0.9518432 (1200)	total: 4m 56s	remaining: 32m 2s
1400:	test: 0.9521804	best: 0.9521804 (1400)	total: 5m 49s	remaining: 31m 34s
1600:	test: 0.9524302	best: 0.9524302 (1600)	total: 6m 42s	remaining: 30m 57s
1800:	test: 0.9526328	best: 0.9526333 (1799)	total: 7m 34s	remaining: 30m 17s
2000:	test: 0.9528000	best: 0.9528007 (1999)	total: 8m 28s	remaining: 29m 38s
2200:	test: 0.9529361	best: 0.9529361 (2200)	total: 9m 21s	remaining: 28m 53s
2400:	test: 0.9530369	best: 0.9530369 (2400)	total: 10m 13s	remaining: 28m 6s
2600

In [12]:
# ============================================================
# CV result
# ============================================================

print_section("CV Result")

cv_auc = roc_auc_score(y_comp, oof)
cv_logloss = log_loss(y_comp, clip_pred(oof))

fold_df = pd.DataFrame(fold_records)

print(f"Overall CV AUC    : {cv_auc:.9f}")
print(f"Overall CV LogLoss: {cv_logloss:.9f}")
print("fold auc mean:", fold_df["auc"].mean())
print("fold auc std :", fold_df["auc"].std())
print("used devices :", sorted(set(used_devices)))

print(fold_df)


CV Result
Overall CV AUC    : 0.953361477
Overall CV LogLoss: 0.260605062
fold auc mean: 0.9533713576817572
fold auc std : 0.0008231311147775614
used devices : ['GPU']
   fold       auc   logloss  best_iteration used_device  n_train_comp  \
0     1  0.954330  0.259582            8191         GPU        351312   
1     2  0.952483  0.262651            7608         GPU        351312   
2     3  0.953487  0.262518            7311         GPU        351312   
3     4  0.952581  0.262292            7359         GPU        351312   
4     5  0.953975  0.255982            8439         GPU        351312   

   n_train_orig  n_valid  n_features  n_cat_features  
0        101371    87828         337              93  
1        101371    87828         337              93  
2        101371    87828         337              93  
3        101371    87828         337              93  
4        101371    87828         337              93  


In [13]:
# ============================================================
# Save artifacts
# ============================================================

print_section("Save Artifacts")

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", pred)

oof_csv = pd.DataFrame({
    CFG.ID_COL: train_ids,
    CFG.TARGET: y_comp.values,
    "pred": oof,
})
oof_csv.to_csv(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv", index=False)

sub = sample_submission.copy()
sub[CFG.TARGET] = clip_pred(pred)

sub_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub.to_csv(sub_path, index=False)

fold_df.to_csv(CFG.OUTDIR / "fold_scores.csv", index=False)

fi_all = pd.concat(feature_importance_records, axis=0, ignore_index=True)
fi_summary = (
    fi_all
    .groupby("feature", as_index=False)["importance"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("mean", ascending=False)
)
fi_all.to_csv(CFG.OUTDIR / "feature_importance_by_fold.csv", index=False)
fi_summary.to_csv(CFG.OUTDIR / "feature_importance.csv", index=False)

feature_df = pd.DataFrame({
    "feature": common_features,
    "is_cat": [c in cat_cols for c in common_features],
    "is_count": [c.endswith("_count") for c in common_features],
    "is_freq": [c.endswith("_freq") for c in common_features],
    "is_digit": [
        ("_int_digit_" in c) or ("_dec_digit_" in c) or ("_sig_" in c)
        for c in common_features
    ],
    "is_string_precision": [
        c in ["RaceProgress_str", "EstimatedTotalLaps_str", "TyreAgeRatio_str"]
        for c in common_features
    ],
    "is_group_stat": [
        ("_mean_by_" in c) or ("_std_by_" in c) or ("_diff_mean_by_" in c)
        for c in common_features
    ],
    "is_race_year_group_stat": [
        is_race_year_groupstat_feature(c)
        for c in common_features
    ],
    "contains_year": ["Year" in c for c in common_features],
    "contains_race_year": ["Race_Year" in c for c in common_features],
    "is_shared001v2_042": [
        c.endswith("_floor_cat_042")
        or c in [
            "PitStop_cat_",
            "Race_Year_",
            "Race_Compound_",
            "RaceProgress_200_quantile_bin_",
            "LapTime (s)_7_quantile_bin_",
        ]
        or c.startswith("_PitStop_cat_")
        or c.startswith("_Race_Year__")
        or c.startswith("_Race_Compound__")
        or c.startswith("_RaceProgress_200_quantile_bin__")
        or c.startswith("_LapTime_s_7_quantile_bin__")
        for c in common_features
    ],
})
feature_df["is_optuna_refit_046"] = False
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

pd.DataFrame({
    "dropped_feature": dropped_groupstat_features_015,
    "reason": [
        "Inherited 015/042 policy: remove Race_Year group-stat feature"
        for _ in dropped_groupstat_features_015
    ],
}).to_csv(CFG.OUTDIR / "dropped_features_race_year_groupstats.csv", index=False)

assert "Race_Year" in common_features
assert "IsOriginalData" in common_features
assert "TyreAgeRatio_str" in common_features
assert not any(is_race_year_groupstat_feature(c) for c in common_features)
assert not feature_df["is_race_year_group_stat"].any()

cat_feature_df = pd.DataFrame({
    "cat_feature": cat_cols,
    "index": cat_features,
})
cat_feature_df.to_csv(CFG.OUTDIR / "categorical_features.csv", index=False)

assert "Race_Year" in cat_cols
assert "IsOriginalData" in common_features
assert "TyreAgeRatio_str" in cat_cols

# Effective params after applying 045 Optuna best params.
effective_cat_params = get_catboost_params(
    seed=CFG.SEED,
    iterations=CFG.ITERATIONS,
    task_type=CFG.TASK_TYPE,
)
effective_cat_params_for_json = {
    k: v
    for k, v in effective_cat_params.items()
    if k not in ["verbose"]
}

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "type": "optuna_refit",
        "created_at": "2026-05-20",
    },
    "base": {
        "model_structure": "exp_20260520_042_cat_shared001v2_features_min",
        "params_source": "exp_20260520_045_cat_shared001v2_features_optuna_search_min",
        "best_params_path": CFG.BEST_PARAMS_PATH,
        "baseline_042_cv_auc": 0.953240915414691,
        "baseline_042_public_lb": 0.95283,
        "optuna_045_best_cv_auc": 0.953352253739158,
        "note": [
            "046 keeps the exact 042 feature engineering, folds, original-row policy, and guardrails.",
            "Only CatBoost hyperparameters are replaced by 045 Optuna best params.",
            "039 LOG features are not added.",
            "No new feature engineering is added in 046.",
        ],
    },
    "optuna": {
        "best_params": CFG.BEST_PARAMS,
        "best_params_source_raw": CFG.BEST_PARAMS_SOURCE_RAW,
        "search_experiment": "exp_20260520_045_cat_shared001v2_features_optuna_search_min",
        "search_baseline": "exp_20260520_042_cat_shared001v2_features_min",
    },
    "data": {
        "train_shape": list(train_raw.shape),
        "test_shape": list(test_raw.shape),
        "original_available": original_raw is not None,
        "original_shape_after_drop": list(original_raw.shape) if original_raw is not None else None,
        "target": CFG.TARGET,
        "id_col": CFG.ID_COL,
        "danger_col": CFG.DANGER_COL,
        "danger_col_used": False,
        "use_original_rows": CFG.USE_ORIGINAL,
        "competition_target_mean": float(train_raw[CFG.TARGET].mean()),
        "original_target_mean": float(original_raw[CFG.TARGET].mean()) if original_raw is not None else None,
    },
    "features": {
        "n_features": int(len(common_features)),
        "n_cat_features": int(len(cat_cols)),
        "dropped_race_year_groupstat_features": dropped_groupstat_features_015,
        "dropped_race_year_groupstat_feature_count": int(len(dropped_groupstat_features_015)),
        "race_year_used": "Race_Year" in common_features,
        "isoriginaldata_used": "IsOriginalData" in common_features,
        "tyreageratio_str_used": "TyreAgeRatio_str" in common_features,
        "race_year_groupstats_used": any(
            is_race_year_groupstat_feature(c) for c in common_features
        ),
        "shared001v2_042_feature_count": int(feature_df["is_shared001v2_042"].sum()),
        "feature_blocks": [
            "domain_features",
            "digit_features",
            "float_signature_features",
            "string_precision_features",
            "shared001v2_style_features_042",
            "frequency_features",
            "shared001v2_count_features_042",
            "light_group_stats_without_race_year_groupstats",
        ],
        "normalized_tyrelife_policy": [
            "Normalized_TyreLife is dropped.",
            "Direct use is forbidden.",
            "Intentional proxy reconstruction is not added.",
        ],
    },
    "model": {
        "family": "CatBoost",
        "estimator": "CatBoostClassifier",
        "params_effective": effective_cat_params_for_json,
        "params_from_045": CFG.BEST_PARAMS,
        "fixed_params": {
            "iterations": CFG.ITERATIONS,
            "bootstrap_type": CFG.BOOTSTRAP_TYPE,
            "subsample": CFG.SUBSAMPLE,
            "loss_function": "Logloss",
            "eval_metric": "AUC",
            "auto_class_weights": CFG.AUTO_CLASS_WEIGHTS,
            "early_stopping_rounds": CFG.EARLY_STOPPING_ROUNDS,
            "task_type_requested": CFG.TASK_TYPE,
            "devices_requested": CFG.DEVICES,
            "devices_used": sorted(set(used_devices)),
        },
        "cv": {
            "scheme": "StratifiedKFold",
            "n_splits": CFG.N_SPLITS,
            "shuffle": True,
            "random_state": CFG.SEED,
        },
    },
    "result": {
        "cv_auc": float(cv_auc),
        "cv_logloss": float(cv_logloss),
        "fold_auc_mean": float(fold_df["auc"].mean()),
        "fold_auc_std": float(fold_df["auc"].std()),
        "fold_scores": fold_records,
        "comparison_to_042": {
            "baseline_cv_auc": 0.953240915414691,
            "delta_cv_auc": float(cv_auc - 0.953240915414691),
            "baseline_public_lb": 0.95283,
            "public_lb": None,
        },
        "comparison_to_045_search": {
            "optuna_best_cv_auc": 0.953352253739158,
            "delta_cv_auc": float(cv_auc - 0.953352253739158),
        },
    },
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "oof_npy": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred_npy": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "oof_csv": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv"),
        "submission": str(sub_path),
        "fold_scores": str(CFG.OUTDIR / "fold_scores.csv"),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "categorical_features": str(CFG.OUTDIR / "categorical_features.csv"),
        "feature_importance": str(CFG.OUTDIR / "feature_importance.csv"),
        "cv_result": str(CFG.OUTDIR / "cv_result.json"),
        "dropped_features": str(CFG.OUTDIR / "dropped_features_race_year_groupstats.csv"),
        "memo_draft": str(CFG.OUTDIR / "memo_draft.yml"),
    },
    "success_criteria": {
        "single": {
            "cv_must_exceed_042": 0.953240915414691,
            "public_lb_must_exceed_042": 0.95283,
            "strong_if_cv_at_least": 0.95335,
        },
        "blend": {
            "nm_cv_must_exceed": 0.9550053246176218,
            "nm_public_lb_must_be_at_least": 0.95431,
            "logreg_public_lb_must_exceed": 0.95430,
        },
    },
    "decision_policy": {
        "single_model": [
            "Compare 046 against 042 single CV/LB.",
            "Single CV improvement alone is not enough for final promotion.",
            "Single CV and Public LB both improving means proceed to add046 blend as a serious candidate.",
        ],
        "blend": [
            "Add 046 to the current standard/pruned stack and compare NM/LogReg against add044 and add042.",
            "Promote only if add046 NM or LogReg improves both CV and Public LB, or maintains Public best with structurally better risk.",
        ],
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------
# memo_draft.yml
# ------------------------------------------------------------
memo_draft = f"""experiment:
  id: {CFG.EXP_ID}
  competition: {CFG.COMPETITION}
  metric: AUC
  created_at: 2026-05-20
  type: optuna_refit
  status: completed

objective:
  summary: >
    045 Optuna searchで得たbest CatBoost paramsを、042 CatBoost shared001v2 features構成へ適用し、
    正式な5fold OOF/pred/submissionを作成する。
    046では特徴量・fold・original rows append・禁止列処理は042から変更しない。
  purpose:
    - 042単体CV/LBをOptuna paramsで更新できるか確認する
    - add046 blendでadd044_nm / add042_logregを置換できるか確認する
  not_purpose:
    - 新規特徴量追加
    - 039 LOG特徴の追加
    - pseudo label
    - stack構成変更

base:
  model_structure: exp_20260520_042_cat_shared001v2_features_min
  params_source: exp_20260520_045_cat_shared001v2_features_optuna_search_min
  baseline_042:
    cv_auc: 0.953240915414691
    public_lb: 0.95283
  optuna_045:
    best_cv_auc: 0.953352253739158
    best_params_path: {CFG.BEST_PARAMS_PATH}

best_params_from_045:
"""
for k, v in CFG.BEST_PARAMS.items():
    memo_draft += f"  {k}: {v}\n"

memo_draft += f"""
fixed_from_042:
  feature_set: same_as_042
  folds: same_as_042
  original_rows: same_as_042
  target: {CFG.TARGET}
  metric: AUC
  forbidden:
    - Normalized_TyreLife
    - Normalized_TyreLife proxy
    - future endpoint proxy
    - pseudo label
    - new feature engineering
  inherited_policy:
    - Race_Year group-stat features are dropped
    - Race_Year itself is kept
    - IsOriginalData is kept
    - TyreAgeRatio_str is kept
    - 039 LOG features are not added

data:
  train_shape: {list(train_raw.shape)}
  test_shape: {list(test_raw.shape)}
  original_available: {original_raw is not None}
  original_shape_after_drop: {list(original_raw.shape) if original_raw is not None else None}
  target: {CFG.TARGET}
  use_original_rows: {CFG.USE_ORIGINAL}

features:
  n_features: {int(len(common_features))}
  n_cat_features: {int(len(cat_cols))}
  shared001v2_042_feature_count: {int(feature_df["is_shared001v2_042"].sum())}
  dropped_race_year_groupstat_feature_count: {int(len(dropped_groupstat_features_015))}
  race_year_used: {"Race_Year" in common_features}
  race_year_groupstats_used: {any(is_race_year_groupstat_feature(c) for c in common_features)}
  isoriginaldata_used: {"IsOriginalData" in common_features}
  tyreageratio_str_used: {"TyreAgeRatio_str" in common_features}

model:
  family: CatBoost
  params_effective:
"""
for k, v in effective_cat_params_for_json.items():
    memo_draft += f"    {k}: {v}\n"

memo_draft += f"""  cv:
    scheme: StratifiedKFold
    n_splits: {CFG.N_SPLITS}
    shuffle: true
    random_state: {CFG.SEED}

result:
  cv_auc: {float(cv_auc)}
  cv_logloss: {float(cv_logloss)}
  fold_auc_mean: {float(fold_df["auc"].mean())}
  fold_auc_std: {float(fold_df["auc"].std())}
  public_lb: null
  comparison_to_042:
    baseline_cv_auc: 0.953240915414691
    delta_cv_auc: {float(cv_auc - 0.953240915414691)}
    baseline_public_lb: 0.95283
  comparison_to_045_search:
    optuna_best_cv_auc: 0.953352253739158
    delta_cv_auc: {float(cv_auc - 0.953352253739158)}

outputs:
  oof_npy: oof_{CFG.EXP_ID}.npy
  pred_npy: pred_{CFG.EXP_ID}.npy
  submission: submission_{CFG.EXP_ID}.csv
  cv_result: cv_result.json
  memo_draft: memo_draft.yml
  feature_columns: feature_columns.csv
  feature_importance: feature_importance.csv
  categorical_features: categorical_features.csv

success_criteria:
  single:
    cv_must_exceed_042: 0.953240915414691
    public_lb_must_exceed_042: 0.95283
    strong_if_cv_at_least: 0.95335
  blend:
    nm_cv_must_exceed: 0.9550053246176218
    nm_public_lb_must_be_at_least: 0.95431
    logreg_public_lb_must_exceed: 0.95430

judgement:
  decision: pending
  notes: >
    まず046単体CVとPublic LBを042と比較する。
    単体CVだけ改善しても採用しない。
    次にadd046 blendでadd044_nmおよびadd042_logregを超えるか確認する。
    CVだけでなくPublic LBとstack内重みを必ず確認する。
"""

with open(CFG.OUTDIR / "memo_draft.yml", "w", encoding="utf-8") as f:
    f.write(memo_draft)

print("Saved to:", CFG.OUTDIR)
print("Submission:", sub_path)
print(json.dumps(result, ensure_ascii=False, indent=2))

print_section("Top Feature Importance")
display(fi_summary.head(80))

print_section("Submission Preview")
display(sub.head())



Save Artifacts
Saved to: /kaggle/working/exp_20260520_046_cat_shared001v2_features_optuna_refit_min
Submission: /kaggle/working/exp_20260520_046_cat_shared001v2_features_optuna_refit_min/submission_exp_20260520_046_cat_shared001v2_features_optuna_refit_min.csv
{
  "experiment": {
    "id": "exp_20260520_046_cat_shared001v2_features_optuna_refit_min",
    "competition": "PS S6E5 Predicting F1 Pit Stops",
    "metric": "AUC",
    "type": "optuna_refit",
    "created_at": "2026-05-20"
  },
  "base": {
    "model_structure": "exp_20260520_042_cat_shared001v2_features_min",
    "params_source": "exp_20260520_045_cat_shared001v2_features_optuna_search_min",
    "best_params_path": "/kaggle/input/datasets/mizushimatoshihiko/npy-files/exp_20260520_045_cat_shared001v2_features_optuna_search_min/best_params_045.json",
    "baseline_042_cv_auc": 0.953240915414691,
    "baseline_042_public_lb": 0.95283,
    "optuna_045_best_cv_auc": 0.953352253739158,
    "note": [
      "046 keeps the exact 042 

,index,feature,mean,std
95,95,IsOriginalData,4.493267,0.158134
242,242,Race_Compound_Stint,3.654062,0.207650
301,301,Year_floor_cat_042,3.474766,0.428037
275,275,TyreAgeRatio_str,2.948082,0.167633
247,247,Race_Year,2.599687,0.180137
...,...,...,...,...
117,117,LapTime (s)_7_quantile_bin_,0.425134,0.125682
68,68,DeltaPerTyreLap_sig_2,0.422290,0.045659
74,74,Driver_Compound_count,0.418985,0.021913
240,240,Race_Compound,0.416695,0.036531



Submission Preview


,id,PitNextLap
0,439140,0.022361
1,439141,0.018778
2,439142,0.015711
3,439143,0.477897
4,439144,0.946221
